<a href="https://colab.research.google.com/github/elenakengne/justeprix/blob/main/task_vectors_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

BASE_DIR = "/content/drive/MyDrive/task_vectors_project"
CHECKPOINTS_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("✅ Dossiers OK")

✅ Dossiers OK


In [7]:
!pip install open_clip_torch==2.0.2 --quiet
!pip install datasets transformers --quiet

import torch
import pkg_resources
print(f"✅ PyTorch {torch.__version__}")
print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
print(f"✅ open_clip : {pkg_resources.get_distribution('open_clip_torch').version}")

/tmp/ipykernel_29466/4143352554.py:5: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


✅ PyTorch 2.11.0+cu128
✅ GPU : Tesla T4
✅ open_clip : 2.0.2


In [8]:
import os

REPO_DIR = "/content/task_vectors_repo"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mlfoundations/task_vectors.git {REPO_DIR}
    print("✅ Repo cloné")
else:
    print("⏭️ Repo déjà présent")

⏭️ Repo déjà présent


In [9]:
CKPT_BASE = "/content/drive/MyDrive/task_vectors_project/checkpoints/task_vectors_checkpoints/ViT-B-32"

PRETRAINED = os.path.join(CKPT_BASE, "zeroshot.pt")

FINETUNED = {
    "Cars":     os.path.join(CKPT_BASE, "Cars/finetuned.pt"),
    "DTD":      os.path.join(CKPT_BASE, "DTD/finetuned.pt"),
    "EuroSAT":  os.path.join(CKPT_BASE, "EuroSAT/finetuned.pt"),
    "GTSRB":    os.path.join(CKPT_BASE, "GTSRB/finetuned.pt"),
    "MNIST":    os.path.join(CKPT_BASE, "MNIST/finetuned.pt"),
    "RESISC45": os.path.join(CKPT_BASE, "RESISC45/finetuned.pt"),
    "SUN397":   os.path.join(CKPT_BASE, "SUN397/finetuned.pt"),
    "SVHN":     os.path.join(CKPT_BASE, "SVHN/finetuned.pt"),
}

print(f"{'zeroshot.pt':<30} {'✅' if os.path.exists(PRETRAINED) else '❌'}")
for name, path in FINETUNED.items():
    print(f"{'finetuned_' + name:<30} {'✅' if os.path.exists(path) else '❌'}")

zeroshot.pt                    ✅
finetuned_Cars                 ✅
finetuned_DTD                  ✅
finetuned_EuroSAT              ✅
finetuned_GTSRB                ✅
finetuned_MNIST                ✅
finetuned_RESISC45             ✅
finetuned_SUN397               ✅
finetuned_SVHN                 ✅


In [10]:
import sys
import torch
import functools

# Ajoute le repo ET son parent pour que 'src' soit trouvable
sys.path.insert(0, "/content/task_vectors_repo/src")
sys.path.insert(0, "/content/task_vectors_repo")

# Patch torch.load
_orig = torch.load
torch.load = functools.partial(_orig, weights_only=False)

print("✅ sys.path OK")
print("✅ Patch torch.load OK")

✅ sys.path OK
✅ Patch torch.load OK


In [11]:
from task_vectors import TaskVector

print("⏳ Chargement MNIST...")
tv_mnist = TaskVector(
    pretrained_checkpoint=PRETRAINED,
    finetuned_checkpoint=FINETUNED["MNIST"]
)
print(f"✅ Task Vector MNIST créé ! ({len(tv_mnist.vector)} couches)")

⏳ Chargement MNIST...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


✅ Task Vector MNIST créé ! (158 couches)


In [12]:
print("⏳ Création de tous les Task Vectors...")

task_vectors = {}
for name, path in FINETUNED.items():
    tv = TaskVector(pretrained_checkpoint=PRETRAINED, finetuned_checkpoint=path)
    task_vectors[name] = tv
    norm = sum(v.norm().item() for v in tv.vector.values())
    print(f"  ✅ TV {name:<12} | {len(tv.vector)} couches | norme totale : {norm:.2f}")

print(f"\n✅ {len(task_vectors)} Task Vectors créés !")

⏳ Création de tous les Task Vectors...
  ✅ TV Cars         | 158 couches | norme totale : 21.06
  ✅ TV DTD          | 158 couches | norme totale : 18.99
  ✅ TV EuroSAT      | 158 couches | norme totale : 17.41
  ✅ TV GTSRB        | 158 couches | norme totale : 18.13
  ✅ TV MNIST        | 158 couches | norme totale : 18.88
  ✅ TV RESISC45     | 158 couches | norme totale : 19.44
  ✅ TV SUN397       | 158 couches | norme totale : 21.93
  ✅ TV SVHN         | 158 couches | norme totale : 20.50

✅ 8 Task Vectors créés !


In [13]:
import torch

class TaskSingularVector:
    def __init__(self, task_vector, rank=1):
        """
        Décompose chaque matrice du task vector par SVD tronquée.
        rank : nombre de composantes singulières à garder (1 = TSV pur)
        """
        self.rank = rank
        self.vector = {}
        self.singular_values = {}  # pour analyse

        for key, tensor in task_vector.vector.items():
            if tensor.ndim < 2:
                # Vecteurs 1D (bias, layernorm) — on garde tel quel
                self.vector[key] = tensor.clone()
                self.singular_values[key] = None
            else:
                # Matrices 2D+ — on applique SVD tronquée
                original_shape = tensor.shape
                mat = tensor.reshape(tensor.shape[0], -1).float()

                U, S, Vh = torch.linalg.svd(mat, full_matrices=False)

                # Tronque au rang k
                U_k  = U[:, :rank]
                S_k  = S[:rank]
                Vh_k = Vh[:rank, :]

                # Reconstruit la matrice approximée
                approx = (U_k * S_k) @ Vh_k
                self.vector[key] = approx.reshape(original_shape)
                self.singular_values[key] = S_k

        print(f"✅ TSV créé (rank={rank}) | {len(self.vector)} couches")

    def apply_to(self, pretrained_checkpoint, scaling_coef=1.0):
        """Applique le TSV au modèle pré-entraîné."""
        with torch.no_grad():
            pretrained_model = torch.load(pretrained_checkpoint)
            new_state_dict = {}
            pretrained_state_dict = pretrained_model.state_dict()
            for key in pretrained_state_dict:
                if key not in self.vector:
                    continue
                new_state_dict[key] = pretrained_state_dict[key] + scaling_coef * self.vector[key]
        pretrained_model.load_state_dict(new_state_dict, strict=False)
        return pretrained_model

    def __neg__(self):
        """Negate le TSV — base de ta contribution originale."""
        neg = TaskSingularVector.__new__(TaskSingularVector)
        neg.rank = self.rank
        neg.singular_values = self.singular_values
        neg.vector = {k: -v for k, v in self.vector.items()}
        return neg

In [14]:
# Crée le TSV pour MNIST (rank=1 = une seule composante singulière)
print("⏳ Création TSV MNIST (rank=1)...")
tsv_mnist_r1 = TaskSingularVector(task_vectors["MNIST"], rank=1)

# Compare TV vs TSV
tv_vec = task_vectors["MNIST"].vector
tsv_vec = tsv_mnist_r1.vector

print("\n📊 Comparaison TV vs TSV (rank=1) sur 3 couches matrices :")
for key in tv_vec:
    if tv_vec[key].ndim >= 2:
        tv_norm  = tv_vec[key].norm().item()
        tsv_norm = tsv_vec[key].norm().item()
        ratio    = tsv_norm / tv_norm * 100
        print(f"  {key[-40:]:<40} | TV: {tv_norm:.4f} | TSV: {tsv_norm:.4f} | ratio: {ratio:.1f}%")
        # Affiche seulement 3 lignes pour ne pas spammer
        break

# Résumé global
tv_total  = sum(v.norm().item() for v in tv_vec.values())
tsv_total = sum(v.norm().item() for v in tsv_vec.values())
print(f"\n  Norme totale TV  : {tv_total:.2f}")
print(f"  Norme totale TSV : {tsv_total:.2f}")
print(f"  Compression      : {tsv_total/tv_total*100:.1f}% de l'information conservée")

⏳ Création TSV MNIST (rank=1)...
✅ TSV créé (rank=1) | 158 couches

📊 Comparaison TV vs TSV (rank=1) sur 3 couches matrices :
  model.positional_embedding               | TV: 0.0015 | TSV: 0.0009 | ratio: 61.9%

  Norme totale TV  : 18.88
  Norme totale TSV : 8.99
  Compression      : 47.6% de l'information conservée


In [15]:
# TSV rank=1 pour tous les datasets
print("⏳ Création TSV (rank=1) pour tous les datasets...\n")

tsv_r1 = {}
for name, tv in task_vectors.items():
    tsv = TaskSingularVector(tv, rank=1)
    tsv_r1[name] = tsv

# Tableau comparatif TV vs TSV
print("\n📊 Résumé TV vs TSV (rank=1) :")
print(f"  {'Dataset':<12} | {'Norme TV':>10} | {'Norme TSV':>10} | {'Compression':>12}")
print("  " + "-"*52)
for name in task_vectors:
    tv_norm  = sum(v.norm().item() for v in task_vectors[name].vector.values())
    tsv_norm = sum(v.norm().item() for v in tsv_r1[name].vector.values())
    ratio    = tsv_norm / tv_norm * 100
    print(f"  {name:<12} | {tv_norm:>10.2f} | {tsv_norm:>10.2f} | {ratio:>11.1f}%")

⏳ Création TSV (rank=1) pour tous les datasets...

✅ TSV créé (rank=1) | 158 couches
✅ TSV créé (rank=1) | 158 couches
✅ TSV créé (rank=1) | 158 couches
✅ TSV créé (rank=1) | 158 couches
✅ TSV créé (rank=1) | 158 couches
✅ TSV créé (rank=1) | 158 couches
✅ TSV créé (rank=1) | 158 couches
✅ TSV créé (rank=1) | 158 couches

📊 Résumé TV vs TSV (rank=1) :
  Dataset      |   Norme TV |  Norme TSV |  Compression
  ----------------------------------------------------
  Cars         |      21.06 |       7.15 |        33.9%
  DTD          |      18.99 |       6.91 |        36.4%
  EuroSAT      |      17.41 |       7.62 |        43.7%
  GTSRB        |      18.13 |       8.04 |        44.4%
  MNIST        |      18.88 |       8.99 |        47.6%
  RESISC45     |      19.44 |       7.24 |        37.2%
  SUN397       |      21.93 |       6.52 |        29.7%
  SVHN         |      20.50 |       9.41 |        45.9%


In [16]:
print("="*55)
print("  CONTRIBUTION ORIGINALE : TSV NEGATION")
print("="*55)

# Négation TV classique (baseline)
neg_tv_mnist = -task_vectors["MNIST"]  # opérateur __neg__ du repo original

# Négation TSV (ta contribution)
neg_tsv_mnist_r1 = -tsv_r1["MNIST"]   # opérateur __neg__ qu'on a implémenté

# Vérifie que la négation est bien appliquée
key_sample = [k for k in neg_tv_mnist.vector if neg_tv_mnist.vector[k].ndim >= 2][0]

tv_val  = task_vectors["MNIST"].vector[key_sample].flatten()[0].item()
neg_tv_val  = neg_tv_mnist.vector[key_sample].flatten()[0].item()
neg_tsv_val = neg_tsv_mnist_r1.vector[key_sample].flatten()[0].item()

print(f"\nCouche : {key_sample}")
print(f"  TV original      :  {tv_val:.6f}")
print(f"  TV nié           :  {neg_tv_val:.6f}  (signe inversé ✅)")
print(f"  TSV nié (rank=1) :  {neg_tsv_val:.6f}  (signe inversé ✅)")

# Compare les normes
tv_norm      = sum(v.norm().item() for v in task_vectors["MNIST"].vector.values())
neg_tv_norm  = sum(v.norm().item() for v in neg_tv_mnist.vector.values())
neg_tsv_norm = sum(v.norm().item() for v in neg_tsv_mnist_r1.vector.values())

print(f"\n📊 Normes :")
print(f"  TV original      : {tv_norm:.2f}")
print(f"  TV nié           : {neg_tv_norm:.2f}  (identique — juste le signe change)")
print(f"  TSV nié (rank=1) : {neg_tsv_norm:.2f}  (compression + négation)")

  CONTRIBUTION ORIGINALE : TSV NEGATION

Couche : model.positional_embedding
  TV original      :  0.000002
  TV nié           :  -0.000002  (signe inversé ✅)
  TSV nié (rank=1) :  0.000000  (signe inversé ✅)

📊 Normes :
  TV original      : 18.88
  TV nié           : 18.88  (identique — juste le signe change)
  TSV nié (rank=1) : 8.99  (compression + négation)


In [17]:
import torchvision.transforms as transforms
import torchvision.datasets as dsets
import torch.nn.functional as F

# Charge quelques images MNIST
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

mnist_test = dsets.MNIST(root='/content/mnist_data', train=False,
                          download=True, transform=transform)
loader = torch.utils.data.DataLoader(mnist_test, batch_size=32, shuffle=False)

print("✅ MNIST chargé —", len(mnist_test), "images de test")

✅ MNIST chargé — 10000 images de test


In [18]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Charge la classification head MNIST
CKPT_BASE = "/content/drive/MyDrive/task_vectors_project/checkpoints/task_vectors_checkpoints/ViT-B-32"
head_mnist = torch.load(f"{CKPT_BASE}/head_MNIST.pt").to(device)
head_mnist.eval()

def evaluate(image_encoder, loader, head, device, max_batches=10):
    """Calcule l'accuracy sur max_batches batches."""
    image_encoder = image_encoder.to(device)
    image_encoder.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            if i >= max_batches:
                break
            images, labels = images.to(device), labels.to(device)
            features = image_encoder(images)
            logits   = head(features)
            preds    = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total * 100

print("⏳ Chargement des 3 modèles et évaluation...\n")

# 1. Modèle finetuned (upper bound)
print("  [1/4] Modèle fine-tuned...")
model_finetuned = torch.load(FINETUNED["MNIST"])
acc_ft = evaluate(model_finetuned, loader, head_mnist, device)
print(f"        Accuracy fine-tuned   : {acc_ft:.1f}%")

# 2. TV nié
print("  [2/4] TV nié...")
model_neg_tv = neg_tv_mnist.apply_to(PRETRAINED, scaling_coef=0.5)
acc_neg_tv = evaluate(model_neg_tv, loader, head_mnist, device)
print(f"        Accuracy TV nié       : {acc_neg_tv:.1f}%")

# 3. TSV nié rank=1
print("  [3/4] TSV nié rank=1...")
model_neg_tsv = neg_tsv_mnist_r1.apply_to(PRETRAINED, scaling_coef=0.5)
acc_neg_tsv = evaluate(model_neg_tsv, loader, head_mnist, device)
print(f"        Accuracy TSV nié r=1  : {acc_neg_tsv:.1f}%")

# 4. Zeroshot (lower bound)
print("  [4/4] Modèle zeroshot...")
model_zero = torch.load(PRETRAINED)
acc_zero = evaluate(model_zero, loader, head_mnist, device)
print(f"        Accuracy zeroshot     : {acc_zero:.1f}%")

print(f"""
╔══════════════════════════════════════╗
║     RÉSULTATS TSV NEGATION — MNIST   ║
╠══════════════════════════════════════╣
║  Fine-tuned (upper bound) : {acc_ft:>5.1f}%  ║
║  Zeroshot   (lower bound) : {acc_zero:>5.1f}%  ║
║  TV nié     (baseline)    : {acc_neg_tv:>5.1f}%  ║
║  TSV nié    (rank=1)      : {acc_neg_tsv:>5.1f}%  ║
╚══════════════════════════════════════╝
""")

⏳ Chargement des 3 modèles et évaluation...

  [1/4] Modèle fine-tuned...
        Accuracy fine-tuned   : 100.0%
  [2/4] TV nié...
        Accuracy TV nié       : 13.1%
  [3/4] TSV nié rank=1...
        Accuracy TSV nié r=1  : 23.4%
  [4/4] Modèle zeroshot...
        Accuracy zeroshot     : 44.4%

╔══════════════════════════════════════╗
║     RÉSULTATS TSV NEGATION — MNIST   ║
╠══════════════════════════════════════╣
║  Fine-tuned (upper bound) : 100.0%  ║
║  Zeroshot   (lower bound) :  44.4%  ║
║  TV nié     (baseline)    :  13.1%  ║
║  TSV nié    (rank=1)      :  23.4%  ║
╚══════════════════════════════════════╝



In [1]:
print("📊 Impact du rang sur la TSV Negation (MNIST)\n")
print(f"  {'Rang':<6} | {'Norme TSV':>10} | {'Compression':>12} | {'Accuracy':>10}")
print("  " + "-"*46)

results_by_rank = {}
for rank in [1, 2, 5, 10, 20, 50]:
    tsv = TaskSingularVector(task_vectors["MNIST"], rank=rank)
    neg_tsv = -tsv
    model = neg_tsv.apply_to(PRETRAINED, scaling_coef=0.5)
    acc = evaluate(model, loader, head_mnist, device)

    tsv_norm = sum(v.norm().item() for v in tsv.vector.values())
    tv_norm  = sum(v.norm().item() for v in task_vectors["MNIST"].vector.values())
    ratio    = tsv_norm / tv_norm * 100

    results_by_rank[rank] = acc
    print(f"  {rank:<6} | {tsv_norm:>10.2f} | {ratio:>11.1f}% | {acc:>9.1f}%")

print(f"\n  TV nié complet (baseline) : 13.1%")
print(f"  Zeroshot                  : 44.4%")

📊 Impact du rang sur la TSV Negation (MNIST)

  Rang   |  Norme TSV |  Compression |   Accuracy
  ----------------------------------------------


NameError: name 'TaskSingularVector' is not defined

In [2]:
# Test rapide — colle dans une nouvelle cellule
try:
    tsv_test = TaskSingularVector(task_vectors["MNIST"], rank=1)
    print("✅ TaskSingularVector disponible")
except NameError:
    print("❌ TaskSingularVector non défini")
try:
    print(f"✅ task_vectors disponible : {list(task_vectors.keys())}")
except NameError:
    print("❌ task_vectors non défini")

❌ TaskSingularVector non défini
❌ task_vectors non défini
